# 02 — The Catalog

After ingest, the Catalog is a **SQLite file** — not a hosted database and not
a live OSM query. This notebook opens `data/catalog.sqlite` if you have one
(your Geofabrik ingest). If not, it builds the committed fixture so the cells
still run.

**Here:** tables, `Catalog.open`, DayHike objects, pandas, Trailhead map.

**Not here:** ingest stages (`01_raw_data_and_ingestion.ipynb`), ranking
(`03_ranking.ipynb`), GeometryStore as GeoPandas (`04a_geometry.ipynb`),
trying new Measures (`04_rank_feature_lab.ipynb`).

A finished Catalog should look like this:

- `IngestReport.kept` equals the number of `dayhikes` rows.
- Every DayHike has a Seed label, a Trailhead, and a length of at least 1 km.
- Kept + unnamed + length + container + node-network drops equals the hiking
  relations loaded at ingest (`route=hiking` or `route=foot`). Unnamed relations are kept
  (`dropped_unnamed` is 0); `name_origin` records where the label came from
  (ADR-0017).
- AmenityFacts are counts inside `AMENITY_RADIUS_M` of the Trailhead.
- DriveTime is **not** a Catalog column; it is a query-time filter (see 03).


In [ ]:
import json
import os
import sqlite3
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

from dayhike import Catalog, build_catalog, geometry_store_path_for
from dayhike.catalog import AMENITY_RADIUS_M, MIN_LENGTH_M, OSM_ATTRIBUTION

local = repo / "data" / "catalog.sqlite"
if local.is_file() and "PYTEST_CURRENT_TEST" not in os.environ:
    path = local
else:
    path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(repo / "tests" / "fixtures" / "catalog.osm", path)

catalog = Catalog.open(path)
conn = sqlite3.connect(path)
meta = dict(conn.execute("SELECT key, value FROM catalog_meta"))
report = json.loads(meta["ingest_report"])
geometry_path = geometry_store_path_for(path)
print("catalog", path)
print("geometry_store", geometry_path, "exists", geometry_path.is_file())
print("attribution", meta.get("attribution"))
print("ingest_report", report)
print("open ok", [hike.name for hike in catalog.dayhikes[:8]])
print("labels table empty", conn.execute("SELECT COUNT(*) FROM labels").fetchone()[0])


## Tables

The Catalog is seven SQLite tables. `PRAGMA table_info` below is the live
schema.

**`catalog_meta`** — key/value. `attribution` is the ODbL line. `ingest_report`
is kept / dropped_unnamed (always 0) / dropped_length / dropped_container /
dropped_node_network (a `network:type=node_network` leg that is not a
RoundTrip) plus NameOrigin counts.

**`dayhikes`** — one DayHike per row.
- `id` — OSM `route=hiking` or `route=foot` relation id.
- `name` — Seed label: OSM `name`; else `ref` with `from – to` or the nearest
  place; else `from – to`; else `Hike near Place`; else `relation:{id}`
  (ADR-0017).
- `name_origin` — `osm_name`, `osm_ref`, `osm_from_to`, `nearest_place`, or
  `synthetic`.
- `length_m` — assembled LineString length, at least 1000 m, no upper cap (ADR-0033).
- `difficulty` — strictest member-way `sac_scale`, or NULL if untagged.

**`trailheads`** — one Trailhead per DayHike.
- `latitude`, `longitude` — WGS84. DriveTime is measured from here at query
  time; it is not a Catalog column.

**`amenity_facts`** — four rows per DayHike (`hut`, `parking`,
`restaurant_cafe`, `drinking_water`).
- `count` — how many of that kind lie inside `AMENITY_RADIUS_M` of the Trailhead.
- `nearest_m` — metres to the closest one, or NULL when count is 0.

**`feature_vectors`** — FeatureVector; `build-similarity` reads it. No DriveTime.
- `length_m`, elevation min/max/gain, `difficulty_ordinal`.
- LandCover fractions: `tree_fraction`, `shrub_fraction`, `grass_fraction`,
  `bare_fraction`, `snow_fraction` (NULL until `sample-landcover`).
- ClimateNormal regime columns: `temperature_c` (June–August mean),
  `precipitation_mm` (mean of the twelve monthly means), `solar_radiation`
  (June–August mean). NULL until `sample-climate`.

**`climate_months`** — hike-level monthly means (month 1–12: `temperature_c`,
`precipitation_mm`, `solar_radiation`). Empty until climate is sampled.
Ranking with a QueryMonth reads this table; Ranking with no month uses the
regime columns.

**`labels`** — Hiker scores. Empty until someone rates a DayHike.


In [ ]:
print(f"sqlite_mb\t{path.stat().st_size / 1e6:.1f}")
tables = [
    row[0]
    for row in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1"
    )
]
for table in tables:
    n_rows = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
    print(f"table\t{table}\t{n_rows}")
    for _cid, col, ctype, notnull, _default, pk in conn.execute(
        f'PRAGMA table_info("{table}")'
    ):
        print(
            f"column\t{table}\t{col}\t{ctype or 'TEXT'}\t"
            f"pk={int(pk)}\tnotnull={int(notnull)}"
        )


## pandas

Europe's Catalog is about 150 MB; that fits in RAM. Each table becomes a
DataFrame with the same name. `play` is a wide join (DayHike + Trailhead +
FeatureVector + pivoted AmenityFacts) for filtering and plotting. Skip the
load if the SQLite file is larger than 200 MB.


In [ ]:
import pandas as pd

SQLITE_PLAY_LIMIT_MB = 200
size_mb = path.stat().st_size / 1e6
print(f"pandas sqlite_mb\t{size_mb:.1f}")
if size_mb > SQLITE_PLAY_LIMIT_MB:
    print("pandas skipped\tSQLite file larger than 200 MB")
    play = pd.DataFrame()
    dayhikes = pd.read_sql_query("SELECT * FROM dayhikes LIMIT 0", conn)
    trailheads = pd.read_sql_query("SELECT * FROM trailheads LIMIT 0", conn)
else:
    catalog_meta = pd.read_sql_query("SELECT * FROM catalog_meta", conn)
    dayhikes = pd.read_sql_query("SELECT * FROM dayhikes", conn)
    trailheads = pd.read_sql_query("SELECT * FROM trailheads", conn)
    amenity_facts = pd.read_sql_query("SELECT * FROM amenity_facts", conn)
    feature_vectors = pd.read_sql_query("SELECT * FROM feature_vectors", conn)
    labels = pd.read_sql_query("SELECT * FROM labels", conn)
    amenity_wide = amenity_facts.pivot(
        index="dayhike_id", columns="kind", values=["count", "nearest_m"]
    )
    amenity_wide.columns = [
        f"{stat}_{kind}" for stat, kind in amenity_wide.columns.to_list()
    ]
    play = (
        dayhikes.rename(columns={"id": "dayhike_id"})
        .merge(trailheads, on="dayhike_id")
        .merge(feature_vectors.drop(columns=["length_m"]), on="dayhike_id")
        .merge(amenity_wide, on="dayhike_id")
    )
    for name, frame in (
        ("catalog_meta", catalog_meta),
        ("dayhikes", dayhikes),
        ("trailheads", trailheads),
        ("amenity_facts", amenity_facts),
        ("feature_vectors", feature_vectors),
        ("labels", labels),
        ("play", play),
    ):
        print(f"pandas {name}\t{frame.shape}")
    print("play dtypes")
    print(play.dtypes.to_string())
    print("play head")
    print(play.head(3).to_string())


## Counts and length

One Trailhead and one FeatureVector per DayHike. Labels stay empty until a
Hiker scores a hike. AmenityFacts store four kinds per DayHike (zero is valid).
DayHikes are at least 1 km (`MIN_LENGTH_M`), with no upper cap (ADR-0033).


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)

n = conn.execute("SELECT COUNT(*) FROM dayhikes").fetchone()[0]
n_th = conn.execute("SELECT COUNT(*) FROM trailheads").fetchone()[0]
n_fv = conn.execute("SELECT COUNT(*) FROM feature_vectors").fetchone()[0]
n_af = conn.execute("SELECT COUNT(*) FROM amenity_facts").fetchone()[0]
n_lab = conn.execute("SELECT COUNT(*) FROM labels").fetchone()[0]
empty_name = conn.execute("SELECT COUNT(*) FROM dayhikes WHERE trim(name) = ''").fetchone()[0]
dropped = (
    report["dropped_unnamed"]
    + report["dropped_length"]
    + report.get("dropped_container", 0)
    + report.get("dropped_node_network", 0)
)
print("dayhikes", n)
print("trailheads", n_th)
print("feature_vectors", n_fv)
print("amenity_fact_rows", n_af)
print("labels", n_lab)
print("kept_plus_dropped", report["kept"] + dropped)
print(f"sanity kept_matches_rows\t{'ok' if report['kept'] == n else 'FAIL'}")
print(f"sanity trailhead_per_dayhike\t{'ok' if n_th == n == n_fv else 'FAIL'}")
print(f"sanity attribution\t{'ok' if meta.get('attribution') == OSM_ATTRIBUTION else 'FAIL'}")
print(f"sanity named\t{'ok' if empty_name == 0 else 'FAIL'}")
print(f"sanity amenity_kinds\t{'ok' if n_af == 4 * n else 'FAIL'}")

lo, hi, avg = conn.execute(
    "SELECT MIN(length_m), MAX(length_m), AVG(length_m) FROM dayhikes"
).fetchone()
short = conn.execute(
    "SELECT COUNT(*) FROM dayhikes WHERE length_m < ?", (MIN_LENGTH_M,)
).fetchone()[0]
print(f"length_m min={lo:.1f} max={hi:.1f} mean={avg:.1f}")
print(
    f"sanity min_length\t"
    f"{'ok' if short == 0 and lo >= MIN_LENGTH_M else 'FAIL'}"
)

lengths_km = [row[0] / 1000 for row in conn.execute("SELECT length_m FROM dayhikes")]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lengths_km, bins=range(1, int(hi / 1000) + 2), color="#0f766e", edgecolor="white")
ax.set_xlabel("length (km)")
ax.set_ylabel("DayHikes")
ax.set_title("Length (1 km and up)")
fig.tight_layout()
show_fig(fig)
print("length_km_histogram")
for km, count in conn.execute(
    "SELECT CAST(length_m / 1000 AS INT), COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 1"
):
    bar = "#" * max(1, round(40 * count / n)) if n else ""
    print(f"  {km:2d} km  {count:6d}  {bar}")


## DayHike objects

`Catalog.open` loads the same rows as Python objects. Each DayHike carries a
Trailhead, Difficulty, AmenityFacts, and a FeatureVector. DriveTime is not on
that vector.


In [ ]:
for hike in catalog.dayhikes[:12]:
    facts = hike.trailhead.amenity_facts
    print(
        hike.id,
        hike.name,
        f"origin={hike.name_origin}",
        f"trailhead=({hike.trailhead.latitude:.4f}, {hike.trailhead.longitude:.4f})",
        f"length_m={hike.length_m:.1f}",
        f"sac_scale={hike.difficulty.sac_scale!r}",
        f"huts={facts.hut.count}",
    )
    print(hike.feature_vector)


## Amenities, Difficulty, elevation

Counts are inside `AMENITY_RADIUS_M` of the Trailhead (20 km). Elevation comes
from `ele` on OSM nodes until `sample-dem`; missing elevation is expected, not
a failed ingest.


In [ ]:
print("amenity_radius_m", AMENITY_RADIUS_M)
print("difficulty")
diff_rows = list(
    conn.execute(
        "SELECT COALESCE(difficulty, 'untagged'), COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 2 DESC"
    )
)
for label, count in diff_rows:
    print(f"  {label}\t{count}")
print("amenities_with_count_gt_0")
amenity_rows = list(
    conn.execute(
        "SELECT kind, SUM(count > 0), round(AVG(count), 1), round(AVG(nearest_m), 0) "
        "FROM amenity_facts GROUP BY kind ORDER BY kind"
    )
)
for kind, positive, avg_n, avg_near in amenity_rows:
    print(f"  {kind}\t{positive}\tmean_count={avg_n}\tmean_nearest_m={avg_near}")
elev_ok, elev_missing = conn.execute(
    "SELECT SUM(elevation_min_m IS NOT NULL), SUM(elevation_min_m IS NULL) FROM feature_vectors"
).fetchone()
print("elevation_present", elev_ok, "missing", elev_missing)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar([row[0] for row in amenity_rows], [row[1] for row in amenity_rows], color="#c2410c")
axes[0].set_title("DayHikes with count > 0")
axes[0].tick_params(axis="x", rotation=15)
axes[1].bar([row[0] for row in diff_rows], [row[1] for row in diff_rows], color="#7c3aed")
axes[1].set_title("Difficulty (sac_scale)")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
show_fig(fig)


## Where the Trailheads are

Boxes overlap on purpose (Alps sit in France and Italy). They are a sketch,
not country membership. On the fixture they collapse into one corner of the
Dinarides; on a Geofabrik Catalog they sketch coverage. Extremes should be
real places in the extract, not null island.


In [ ]:
bbox = conn.execute(
    "SELECT MIN(latitude), MAX(latitude), MIN(longitude), MAX(longitude) FROM trailheads"
).fetchone()
print("trailhead_bbox lat", bbox[0], bbox[1], "lon", bbox[2], bbox[3])
print("sanity not_null_island", "ok" if bbox[0] != 0 or bbox[2] != 0 else "FAIL")

print("extremes")
for label, order in (
    ("west", "t.longitude ASC"),
    ("east", "t.longitude DESC"),
    ("south", "t.latitude ASC"),
    ("north", "t.latitude DESC"),
):
    row = conn.execute(
        "SELECT d.name, round(t.latitude, 4), round(t.longitude, 4) "
        "FROM trailheads t JOIN dayhikes d ON d.id = t.dayhike_id "
        f"ORDER BY {order} LIMIT 1"
    ).fetchone()
    print(f"  {label}\t{row[0]}\t{row[1]}\t{row[2]}")

regions = [
    ("Iberia", 36, 44, -10, 4),
    ("France", 42, 51.5, -5, 8),
    ("Alps", 44, 48.5, 5, 16),
    ("Italy", 36, 47, 6, 19),
    ("Dinaric/Balkans", 40, 47, 13, 24),
    ("Germany/Benelux", 47, 55, 2, 15),
    ("UK/IE", 49, 61, -11, 2),
    ("Scandinavia", 55, 72, 4, 32),
    ("Poland/Baltics", 49, 60, 14, 28),
    ("Romania/Carpathians", 43, 49, 20, 30),
    ("Greece", 34, 42, 19, 29),
    ("Azores/Madeira", 32, 40, -32, -13),
]
print("trailheads_in_overlapping_boxes")
for name, minlat, maxlat, minlon, maxlon in regions:
    count = conn.execute(
        "SELECT COUNT(*) FROM trailheads "
        "WHERE latitude BETWEEN ? AND ? AND longitude BETWEEN ? AND ?",
        (minlat, maxlat, minlon, maxlon),
    ).fetchone()[0]
    print(f"  {name}\t{count}")


## Map: Trailheads

Trailheads on an Esri street map. GeoPandas draws the points; contextily
fetches tiles (needs a network; skipped under pytest or when offline).


In [ ]:
import contextily as cx
import geopandas as gpd

if len(play):
    plot_df = play
else:
    plot_df = pd.read_sql_query(
        '''
        SELECT d.id AS dayhike_id, d.name, t.latitude, t.longitude
        FROM dayhikes d JOIN trailheads t ON t.dayhike_id = d.id
        ''',
        conn,
    )

gdf = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(plot_df.longitude, plot_df.latitude),
    crs="EPSG:4326",
).to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(11, 9))
gdf.plot(ax=ax, markersize=2, alpha=0.35, color="#c2410c", edgecolors="none")
if "PYTEST_CURRENT_TEST" not in os.environ:
    try:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldStreetMap)
    except Exception as exc:  # offline: the points still plot
        print("basemap skipped", type(exc).__name__)
ax.set_title(f"Trailheads ({len(gdf)} of {n})")
ax.set_axis_off()
fig.tight_layout()
show_fig(fig)
print("map trailheads_plotted", len(gdf), "of", n)


## GeometryStore

The sidecar `{catalog}.geometry.sqlite` is written at ingest. Ranking does not
read it. It holds DayHikeGeometry (ordered vertices), member-way tags, and raw
amenity points including Viewpoint, Peak, and Shelter — kinds that are **not**
Catalog AmenityFacts.

Load those tables as GeoPandas in [`04a_geometry.ipynb`](04a_geometry.ipynb).
`build-similarity` turns them into similarity facts; try new ones in
`04_rank_feature_lab.ipynb`.


In [ ]:
print("geometry_store", geometry_path, "exists", geometry_path.is_file())


## Names at the length edges

Shortest and longest kept DayHikes should still read as walks, not as country
crossings. Synthetic labels look like `relation:{id}`; filter on
`name_origin = 'osm_name'` if you want OSM names only. The cell also counts
each NameOrigin with a few examples, so a labelling rule that goes wrong shows
up here.


In [ ]:
print("shortest")
for osm_id, name, length_m in conn.execute(
    "SELECT id, name, round(length_m) FROM dayhikes ORDER BY length_m ASC LIMIT 8"
):
    print(f"  {osm_id}\t{length_m}\t{name}")
print("longest")
for osm_id, name, length_m in conn.execute(
    "SELECT id, name, round(length_m) FROM dayhikes ORDER BY length_m DESC LIMIT 8"
):
    print(f"  {osm_id}\t{length_m}\t{name}")
print("sample")
for name in conn.execute("SELECT name FROM dayhikes ORDER BY random() LIMIT 12"):
    print(f"  {name[0]}")
print("name_origin")
for origin, count in conn.execute(
    "SELECT name_origin, COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 2 DESC"
).fetchall():
    examples = [
        row[0]
        for row in conn.execute(
            "SELECT name FROM dayhikes WHERE name_origin = ? ORDER BY random() LIMIT 3",
            (origin,),
        )
    ]
    print(f"  {origin}\t{count}\t{' | '.join(examples)}")
conn.close()


**Next:** [`03_ranking.ipynb`](03_ranking.ipynb) ranks by Seed on the
Measures. To try a new fact as a Measure, skip to
[`04_rank_feature_lab.ipynb`](04_rank_feature_lab.ipynb).
